# 서울 지하철 유동인구 x 상권 매출 분석
## SCQA 기반 창업 입지 추천 프로젝트

### 분석 개요
- **목적**: 유동인구 많은 역이 실제 매출도 높은 상권인지 검증
- **데이터**: 서울시 지하철 승하차 + 상권분석서비스
- **방법**: 회귀분석, Spearman 상관, t-test, ANOVA


## 0. 환경 설정

In [1]:
import csv, json, re, os, zipfile, io
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind, f_oneway, linregress, spearmanr
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False
print("환경 설정 완료")


환경 설정 완료


## 1. 데이터 로딩

In [2]:
# 지하철 CSV
subway_path = 'CARD_SUBWAY_MONTH_202605.csv'
with open(subway_path, 'r', encoding='utf-8-sig') as f:
    reader = csv.DictReader(f)
    subway_rows = list(reader)
print(f"지하철: {len(subway_rows)}건")

# 상권 ZIP
with zipfile.ZipFile('서울시_상권분석서비스_2025.zip', 'r') as z:
    with z.open(z.namelist()[0]) as f:
        text = f.read().decode('cp949')
all_rows = list(csv.DictReader(io.StringIO(text)))
print(f"상권: {len(all_rows)}건")


FileNotFoundError: [Errno 2] No such file or directory: 'CARD_SUBWAY_MONTH_202605.csv'

## 2. 역별 일평균 승객 계산

In [ ]:
station_totals = {}
for row in subway_rows:
    name, line, b, a = row['역명'], row['노선명'], int(row['승차총승객수']), int(row['하차총승객수'])
    if name not in station_totals:
        station_totals[name] = {'b':0, 'a':0, 'd':0, 'l':line}
    station_totals[name]['b'] += b
    station_totals[name]['a'] += a
    station_totals[name]['d'] += 1

stations = {}
for n, v in station_totals.items():
    avg = (v['b']+v['a'])/v['d']
    ratio = v['a']/(v['b']+v['a'])
    typ = '도착형' if ratio>0.505 else ('출발형' if ratio<0.495 else '균형형')
    stations[n] = {'total':avg, 'line':v['l'], 'type':typ}

print(f"전체: {len(stations)}개 역")
type_cnt = {'도착형':0,'출발형':0,'균형형':0}
for s in stations.values(): type_cnt[s['type']] += 1
print(f"유형분포: {type_cnt}")


## 3. 역명-상권명 매칭

In [ ]:
commercial_match = {}
for r in all_rows:
    m = re.match(r'^(.+?)역', r['상권_코드_명'])
    if m:
        cand = m.group(1).strip()
        for sname in stations:
            clean = re.sub(r'\([^)]*\)', '', sname).strip()
            if clean == cand:
                if sname not in commercial_match:
                    commercial_match[sname] = set()
                commercial_match[sname].add(r['상권_코드_명'])

print(f"매칭: {len(commercial_match)}개 역")


## 4. 매출 연결 및 순위 분석

In [ ]:
def get_sales(sname):
    if sname not in commercial_match: return 0
    best = max(commercial_match[sname],
               key=lambda c: sum(int(r['당월_매출_금액']) for r in all_rows if r['상권_코드_명']==c))
    return sum(int(r['당월_매출_금액']) for r in all_rows if r['상권_코드_명']==best)/1e8

rank_data = []
for sname in commercial_match:
    sales = get_sales(sname)
    if sales > 0 and sname in stations:
        rank_data.append({'name':sname, 'pass':stations[sname]['total'], 'sales':sales})

# 순위 부여
rank_data.sort(key=lambda x: -x['pass'])
for i, d in enumerate(rank_data): d['pass_rank'] = i+1
rank_data.sort(key=lambda x: -x['sales'])
for i, d in enumerate(rank_data): d['sales_rank'] = i+1
for d in rank_data: d['gap'] = d['sales_rank'] - d['pass_rank']

print("유동인구 TOP 10 vs 매출 순위:")
print(f"{'역명':12s} {'승객순위':8s} {'매출순위':8s} {'격차':8s}")
for d in sorted(rank_data, key=lambda x: x['pass_rank'])[:10]:
    sgn = '+' if d['gap']>0 else ''
    print(f"{d['name']:12s} {d['pass_rank']:3d}등      {d['sales_rank']:3d}등      {sgn}{d['gap']}단계")


## 5. 통계분석

In [ ]:
# 5-1. Spearman 상관
rho, p_sp = spearmanr([d['pass_rank'] for d in rank_data],
                      [d['sales_rank'] for d in rank_data])
print(f"Spearman rho = {rho:.3f}, p = {p_sp:.4f}")

# 5-2. 회귀분석
x = [d['pass'] for d in rank_data]
y = [d['sales'] for d in rank_data]
slope, intercept, rv, pv, _ = linregress(x, y)
print(f"Regression: R2 = {rv**2:.3f}, y = {slope:.1f}x + ({intercept:.0f})")

# 5-3. t-test / ANOVA
rank_data.sort(key=lambda x: x['sales_rank'])
n = len(rank_data)
for i, d in enumerate(rank_data):
    d['group'] = '상' if i < n//3 else ('중' if i < n*2//3 else '하')

g_h = [d['pass']/10000 for d in rank_data if d['group']=='상']
g_m = [d['pass']/10000 for d in rank_data if d['group']=='중']
g_l = [d['pass']/10000 for d in rank_data if d['group']=='하']

t, p_tt = ttest_ind(g_h, g_l, equal_var=False)
f, p_an = f_oneway(g_h, g_m, g_l)
print(f"t-test: t={t:.3f}, p={p_tt:.4f}")
print(f"ANOVA: F={f:.3f}, p={p_an:.4f}")


## 6. 시각화

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
x_all = np.array([d['pass']/10000 for d in rank_data])
y_all = np.array([d['sales'] for d in rank_data])
ax.scatter(x_all, y_all, c='#CBD5E1', s=25, alpha=0.7, zorder=1, label='전체 역')

x_top = np.array([d['pass']/10000 for d in rank_data[:10]])
y_top = np.array([min(d['sales'], 20000) for d in rank_data[:10]])
ax.scatter(x_top, y_top, c='#EF4444', s=80, zorder=3, label='TOP10')

sl, ic, rv2, _, _ = linregress(x_all, y_all)
xl = np.linspace(0, 20000/sl, 100)
ax.plot(xl, sl*xl+ic, color='#0F2747', lw=2.5, zorder=2)

ax.set_ylim(0, 20000)
ax.set_xlabel('유동인구 (만명/일)', fontsize=13)
ax.set_ylabel('추정 매출액 (억원)', fontsize=13)
ax.set_title('유동인구와 추정매출의 관계', fontsize=16, fontweight='bold')
ax.grid(alpha=0.15)
ax.legend()
plt.tight_layout()
plt.savefig('scatter_final.png', dpi=150, bbox_inches='tight')
plt.show()


## 7. 결과 요약

| 분석 | 결과 | 의미 |
|------|------|------|
| 회귀분석 | R2 = 0.164 | 유동인구는 매출의 16.4%만 설명 |
| Spearman | rho = 0.574 | 순위 일치도 불완전 |
| t-test | p < 0.001 | 매출 그룹 간 유동인구 차이 유의 |
| ANOVA | F = 43.832, p < 0.001 | 3그룹 간 차이 유의 |

### 결론
> 유동인구가 많은 역이 반드시 살아있는 상권은 아닙니다.
> 진짜 살아있는 상권은 사람이 지나가는 곳이 아니라, 소비가 발생하는 곳입니다.
